In [0]:
%run "/Workspace/Users/iancarodrigues108@gmail.com/ETL_Arquitetura_Medalhao_BOI_GORDO/0.Config/config"


catalogo:  workspace
schemas:  bronze_economia silver_economia gold_economia


In [0]:
#importando biblioteta

from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, regexp_replace, col, greatest # essas informacoes de funcoes da pyspark sql



In [0]:
#Lendo as tabelas ipca e boi gordo

ipca = spark.table(f"{CATALOG}.{BRONZE}.ipca")
boi = spark.table(f"{CATALOG}.{BRONZE}.boi_gordo")

print("Schema IPCA: ")
ipca.printSchema()

print("Schema Boi Gordo: ")
boi.printSchema()

#display(ipca)
#display(boi)



Schema IPCA: 
root
 |-- data: string (nullable = true)
 |-- ipca: double (nullable = true)
 |-- data_coleta: timestamp (nullable = true)

Schema Boi Gordo: 
root
 |-- Data: string (nullable = true)
 |-- Valor: string (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



In [0]:
# VISUALIZAÇÃO DAS TABELAS

display(ipca)
display(boi)

data,ipca,data_coleta
01/01/2025,0.16,2026-07-21T19:13:41.796Z
01/02/2025,1.31,2026-07-21T19:13:41.796Z
01/03/2025,0.56,2026-07-21T19:13:41.796Z
01/04/2025,0.43,2026-07-21T19:13:41.796Z
01/05/2025,0.26,2026-07-21T19:13:41.796Z
01/06/2025,0.24,2026-07-21T19:13:41.796Z
01/07/2025,0.26,2026-07-21T19:13:41.796Z
01/08/2025,-0.11,2026-07-21T19:13:41.796Z
01/09/2025,0.48,2026-07-21T19:13:41.796Z
01/10/2025,0.09,2026-07-21T19:13:41.796Z


Data,Valor,data_coleta
01/2025,"324,95",2026-07-21T19:13:26.427Z
02/2025,"319,21",2026-07-21T19:13:26.427Z
03/2025,"312,47",2026-07-21T19:13:26.427Z
04/2025,"323,96",2026-07-21T19:13:26.427Z
05/2025,"308,15",2026-07-21T19:13:26.427Z
06/2025,"313,51",2026-07-21T19:13:26.427Z
07/2025,"299,97",2026-07-21T19:13:26.427Z
08/2025,"307,25",2026-07-21T19:13:26.427Z
09/2025,"307,87",2026-07-21T19:13:26.427Z
10/2025,"310,51",2026-07-21T19:13:26.427Z


In [0]:
# Renomeando as colunas do Boi Gordo (Data/Valor para data/boi_gordo)

from pyspark.sql.functions import col

boi = (
    boi.withColumnRenamed("Data", "data")
    .withColumnRenamed("Valor", "boi_gordo")
)
display(boi)
boi.printSchema()

data,boi_gordo,data_coleta
01/2025,"324,95",2026-07-21T19:13:26.427Z
02/2025,"319,21",2026-07-21T19:13:26.427Z
03/2025,"312,47",2026-07-21T19:13:26.427Z
04/2025,"323,96",2026-07-21T19:13:26.427Z
05/2025,"308,15",2026-07-21T19:13:26.427Z
06/2025,"313,51",2026-07-21T19:13:26.427Z
07/2025,"299,97",2026-07-21T19:13:26.427Z
08/2025,"307,25",2026-07-21T19:13:26.427Z
09/2025,"307,87",2026-07-21T19:13:26.427Z
10/2025,"310,51",2026-07-21T19:13:26.427Z


root
 |-- data: string (nullable = true)
 |-- boi_gordo: string (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



In [0]:
# Ajustando as datas para o boi gordo (MM/YYYY) para date YYYY-MM como string

# Primeiro importa a funcao to_date e date_format

from pyspark.sql.functions import to_date, date_format

# 1º transforma string "MM/YYYY" em date (com dia default)
boi = boi.withColumn("data", to_date("data","MM/yyyy"))

# 2º transforma date em string YYYY-MM
boi = boi.withColumn("data", date_format("data", "yyyy-MM"))



In [0]:
display(boi)

data,boi_gordo,data_coleta
2025-01,"324,95",2026-07-21T19:13:26.427Z
2025-02,"319,21",2026-07-21T19:13:26.427Z
2025-03,"312,47",2026-07-21T19:13:26.427Z
2025-04,"323,96",2026-07-21T19:13:26.427Z
2025-05,"308,15",2026-07-21T19:13:26.427Z
2025-06,"313,51",2026-07-21T19:13:26.427Z
2025-07,"299,97",2026-07-21T19:13:26.427Z
2025-08,"307,25",2026-07-21T19:13:26.427Z
2025-09,"307,87",2026-07-21T19:13:26.427Z
2025-10,"310,51",2026-07-21T19:13:26.427Z


In [0]:
# Ajustando as datas do IPCA (DD/MM/YYYY) para date YYYY-MM como string

# 1º transforma string "DD/MM/YYYY" em date (com dia default)
ipca = ipca.withColumn("data", to_date("data","dd/MM/yyyy"))

# 2º transforma date em string YYYY-MM
ipca = ipca.withColumn("data", date_format("data", "yyyy-MM"))


In [0]:
display(ipca)

data,ipca,data_coleta
2025-01,0.16,2026-07-22T17:48:56.750Z
2025-02,1.31,2026-07-22T17:48:56.750Z
2025-03,0.56,2026-07-22T17:48:56.750Z
2025-04,0.43,2026-07-22T17:48:56.750Z
2025-05,0.26,2026-07-22T17:48:56.750Z
2025-06,0.24,2026-07-22T17:48:56.750Z
2025-07,0.26,2026-07-22T17:48:56.750Z
2025-08,-0.11,2026-07-22T17:48:56.750Z
2025-09,0.48,2026-07-22T17:48:56.750Z
2025-10,0.09,2026-07-22T17:48:56.750Z


In [0]:
# Fazer um join simples por data 
df_join = (
    ipca.join(boi, on="data", how="inner")
    .select("data", "ipca", "boi_gordo")
)

In [0]:
display(df_join)


data,ipca,boi_gordo
2025-01,0.16,"324,95"
2025-02,1.31,"319,21"
2025-03,0.56,"312,47"
2025-04,0.43,"323,96"
2025-05,0.26,"308,15"
2025-06,0.24,"313,51"
2025-07,0.26,"299,97"
2025-08,-0.11,"307,25"
2025-09,0.48,"307,87"
2025-10,0.09,"310,51"


In [0]:
# Join com alias, trazendo as coletas do ipa e boi gordo

ip = ipca.alias("ip")
bo = boi.alias("bo")

df_join = (
    ip.join(bo, col("ip.data") == col("bo.data"), "inner")
     .select(
         col("ip.data").alias("data"),
         col("ip.ipca").alias("ipca"),
         col("bo.boi_gordo").alias("boi_gordo"),
         col("ip.data_coleta").alias("data_coleta")
    )
)


In [0]:
display(df_join)

data,ipca,boi_gordo,data_coleta
2025-01,0.16,"324,95",2026-07-22T17:48:56.750Z
2025-02,1.31,"319,21",2026-07-22T17:48:56.750Z
2025-03,0.56,"312,47",2026-07-22T17:48:56.750Z
2025-04,0.43,"323,96",2026-07-22T17:48:56.750Z
2025-05,0.26,"308,15",2026-07-22T17:48:56.750Z
2025-06,0.24,"313,51",2026-07-22T17:48:56.750Z
2025-07,0.26,"299,97",2026-07-22T17:48:56.750Z
2025-08,-0.11,"307,25",2026-07-22T17:48:56.750Z
2025-09,0.48,"307,87",2026-07-22T17:48:56.750Z
2025-10,0.09,"310,51",2026-07-22T17:48:56.750Z


In [0]:
# Correcao do tipos data como date YYYY-MM-01, números como double 

from pyspark.sql.functions import regexp_replace

# Data mudar de texto para virar date assumindo dia 01

df_join = df_join.withColumn("data", to_date(col("data"),"yyyy-MM"))

# ipca e o boi gordo: Garantir double (se vier com virgula, trocar por ponto)

df_join = (df_join
           .withColumn("ipca", regexp_replace("ipca",      ",",".").cast("double"))
           .withColumn("boi_gordo", regexp_replace("boi_gordo", ",",".").cast("double"))
)

df_join.printSchema()
display(df_join.orderBy("data").limit(50))

root
 |-- data: date (nullable = true)
 |-- ipca: double (nullable = true)
 |-- boi_gordo: double (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



data,ipca,boi_gordo,data_coleta
2025-01-01,0.16,324.95,2026-07-22T17:48:56.750Z
2025-02-01,1.31,319.21,2026-07-22T17:48:56.750Z
2025-03-01,0.56,312.47,2026-07-22T17:48:56.750Z
2025-04-01,0.43,323.96,2026-07-22T17:48:56.750Z
2025-05-01,0.26,308.15,2026-07-22T17:48:56.750Z
2025-06-01,0.24,313.51,2026-07-22T17:48:56.750Z
2025-07-01,0.26,299.97,2026-07-22T17:48:56.750Z
2025-08-01,-0.11,307.25,2026-07-22T17:48:56.750Z
2025-09-01,0.48,307.87,2026-07-22T17:48:56.750Z
2025-10-01,0.09,310.51,2026-07-22T17:48:56.750Z


In [0]:
# Salvando a tabela Delta Silver (silver_etl_economia)
df_join.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.economia")